In [ ]:
box::use(
  dada = dada2,
  bios = Biostrings,
  stri = stringr,
  msa,
  ape,
  tidyverse[...], ggplot2[...], ggtree[...]
)

In [2]:
wdpath <- getwd()
demult.path <- paste0(wdpath, "/zuzana/demultiplexed_RSII")
fns <- sort(list.files(demult.path, pattern = ".fastq.gz", full.names = TRUE))

In [3]:
filts <- file.path(wdpath, "zuzana/filtered_RSII", basename(fns))

track <- dada$filterAndTrim(
  fns, filts,
  minLen = 2000, # Min length for sequence
  maxLen = 3000, # Max length for sequence
  rm.phix = FALSE, # No phix added (Illumina specific)
  qualityType = "FastqQuality", # Suggested for PacBio
  multithread = TRUE, # Allow multithread
  verbose = FALSE, # Print progress
  maxEE = 2, # Suggested default
  minQ = 20
)
exists <- file.exists(filts)
paste("Sample", basename(filts[!exists]), "did not pass filtering")
write.csv(track, "zuzana/zuzana_RSII_2000_filtering_tracking.csv")

Creating output directory: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered_RSII

Some input samples had no reads pass the filter.



[1] "Sample Rh_Irregularis_Chomutov_5A_cut.fastq.gz did not pass filtering"
[2] "Sample S_cerevisiaejenfwd_cut.fastq.gz did not pass filtering"        
[3] "Sample Sa_cerevisiae_Sa_cerev_cut.fastq.gz did not pass filtering"

In [4]:
filts <- file.path(wdpath, "zuzana/filtered_RSII", basename(fns))
data.frame(
  "barcode" = sapply(
    filts[exists],
    function(x) {stri$str_split_i(basename(x), "[.]", 3)}
  ),
  "median" = sapply(
    filts[exists],
    function(x) {median(bios$width(bios$readDNAStringSet(x ,format='FASTQ')))}
  )
) |> write.csv("zuzana/zuzana_RSII_filtered_medians.csv")

In [5]:
filts <- file.path(wdpath, "zuzana/filtered_RSII", basename(fns))
exists <- file.exists(filts)
filts <- filts[exists]
drp <- dada$derepFastq(filts, verbose = TRUE)

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered_RSII/A_leptoticha_CR312_1A_cut.fastq.gz

Encountered 11 unique sequences from 11 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered_RSII/A_leptoticha_CR312_1B_cut.fastq.gz

Encountered 25 unique sequences from 71 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered_RSII/A_leptoticha_CR312_1C_cut.fastq.gz

Encountered 28 unique sequences from 32 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered_RSII/Ac_koskei_WV685_1A_cut.fastq.gz

Encountered 17 unique sequences from 63 total sequences read.

Dereplicating sequence entries in Fastq file: /home/robert/Public/Github/pacbio-pipeline/zuzana/filtered_RSII/Ac_koskei_WV685_1B_cut.fastq.gz

Encountered 7 unique seq

In [ ]:
err <- dada$learnErrors(
  drp,
  errorEstimationFunction = dada$PacBioErrfun, # Set error estimation function for PacBio 
  multithread = TRUE, # Allow multithread
  BAND_SIZE = 32 # Suggested for Pacbio
)

8244470 total bases in 3527 reads from 87 samples will be used for learning the error rates.


The max qual score of 93 was not detected. Using standard error fitting.

The max qual score of 93 was not detected. Using standard error fitting.

The max qual score of 93 was not detected. Using standard error fitting.

The max qual score of 93 was not detected. Using standard error fitting.



In [7]:
dd <- dada$dada(
  drp,
  err = err,
  BAND_SIZE = 32,
  multithread = TRUE
)

Sample 1 - 11 reads in 11 unique sequences.
Sample 2 - 71 reads in 25 unique sequences.
Sample 3 - 32 reads in 28 unique sequences.
Sample 4 - 63 reads in 17 unique sequences.
Sample 5 - 9 reads in 7 unique sequences.
Sample 6 - 19 reads in 11 unique sequences.
Sample 7 - 4 reads in 4 unique sequences.
Sample 8 - 9 reads in 5 unique sequences.
Sample 9 - 7 reads in 5 unique sequences.
Sample 10 - 26 reads in 16 unique sequences.
Sample 11 - 9 reads in 8 unique sequences.
Sample 12 - 12 reads in 10 unique sequences.
Sample 13 - 76 reads in 26 unique sequences.
Sample 14 - 29 reads in 25 unique sequences.
Sample 15 - 55 reads in 19 unique sequences.
Sample 16 - 53 reads in 13 unique sequences.
Sample 17 - 104 reads in 48 unique sequences.
Sample 18 - 68 reads in 16 unique sequences.
Sample 19 - 5 reads in 4 unique sequences.
Sample 20 - 6 reads in 6 unique sequences.
Sample 21 - 11 reads in 10 unique sequences.
Sample 22 - 5 reads in 5 unique sequences.
Sample 23 - 65 reads in 9 unique s

In [8]:
data.frame(
  "denoised" = sapply(
    dd,
    function(x) {sum(dada$getUniques(x))}
  )
) |> write.csv("zuzana/zuzana_RSII_denoised.csv")

In [9]:
st <- dada$makeSequenceTable(dd)
dim(st)

[1]  87 111

In [10]:
st.nobim <- dada$removeBimeraDenovo(
  st, method = "consensus",
  multithread = TRUE,
  verbose = TRUE
)

Identified 2 bimeras out of 111 input sequences.



In [11]:
data.frame(
  "nobimeras" = rowSums(st.nobim)
) |> write.csv("zuzana/zuzana_RSII_nobim.csv")

In [12]:
asv_table <- t(st.nobim)
colnames(asv_table) <- sapply(strsplit(rownames(st.nobim), "[.]"), `[`, 1)
rep_seqs <- bios$DNAStringSet(colnames(st.nobim))
rownames(asv_table) <- paste0("zuzana-RSII_ASV_", seq(colnames(st.nobim)))
names(rep_seqs) <- rownames(asv_table)

In [13]:
row.names(asv_table)
colnames(asv_table)

[1] "zuzana-RSII_ASV_1"   "zuzana-RSII_ASV_2"   "zuzana-RSII_ASV_3"  
  [4] "zuzana-RSII_ASV_4"   "zuzana-RSII_ASV_5"   "zuzana-RSII_ASV_6"  
  [7] "zuzana-RSII_ASV_7"   "zuzana-RSII_ASV_8"   "zuzana-RSII_ASV_9"  
 [10] "zuzana-RSII_ASV_10"  "zuzana-RSII_ASV_11"  "zuzana-RSII_ASV_12" 
 [13] "zuzana-RSII_ASV_13"  "zuzana-RSII_ASV_14"  "zuzana-RSII_ASV_15" 
 [16] "zuzana-RSII_ASV_16"  "zuzana-RSII_ASV_17"  "zuzana-RSII_ASV_18" 
 [19] "zuzana-RSII_ASV_19"  "zuzana-RSII_ASV_20"  "zuzana-RSII_ASV_21" 
 [22] "zuzana-RSII_ASV_22"  "zuzana-RSII_ASV_23"  "zuzana-RSII_ASV_24" 
 [25] "zuzana-RSII_ASV_25"  "zuzana-RSII_ASV_26"  "zuzana-RSII_ASV_27" 
 [28] "zuzana-RSII_ASV_28"  "zuzana-RSII_ASV_29"  "zuzana-RSII_ASV_30" 
 [31] "zuzana-RSII_ASV_31"  "zuzana-RSII_ASV_32"  "zuzana-RSII_ASV_33" 
 [34] "zuzana-RSII_ASV_34"  "zuzana-RSII_ASV_35"  "zuzana-RSII_ASV_36" 
 [37] "zuzana-RSII_ASV_37"  "zuzana-RSII_ASV_38"  "zuzana-RSII_ASV_39" 
 [40] "zuzana-RSII_ASV_40"  "zuzana-RSII_ASV_41"  "zuzana-RSII_ASV_42" 
 [43] "zuzana-RSII_ASV_43"  "zuzana-RSII_ASV_44"  "zuzana-RSII_ASV_45" 
 [46] "zuzana-RSII_ASV_46"  "zuzana-RSII_ASV_47"  "zuzana-RSII_ASV_48" 
 [49] "zuzana-RSII_ASV_49"  "zuzana-RSII_ASV_50"  "zuzana-RSII_ASV_51" 
 [52] "zuzana-RSII_ASV_52"  "zuzana-RSII_ASV_53"  "zuzana-RSII_ASV_54" 
 [55] "zuzana-RSII_ASV_55"  "zuzana-RSII_ASV_56"  "zuzana-RSII_ASV_57" 
 [58] "zuzana-RSII_ASV_58"  "zuzana-RSII_ASV_59"  "zuzana-RSII_ASV_60" 
 [61] "zuzana-RSII_ASV_61"  "zuzana-RSII_ASV_62"  "zuzana-RSII_ASV_63" 
 [64] "zuzana-RSII_ASV_64"  "zuzana-RSII_ASV_65"  "zuzana-RSII_ASV_66" 
 [67] "zuzana-RSII_ASV_67"  "zuzana-RSII_ASV_68"  "zuzana-RSII_ASV_69" 
 [70] "zuzana-RSII_ASV_70"  "zuzana-RSII_ASV_71"  "zuzana-RSII_ASV_72" 
 [73] "zuzana-RSII_ASV_73"  "zuzana-RSII_ASV_74"  "zuzana-RSII_ASV_75" 
 [76] "zuzana-RSII_ASV_76"  "zuzana-RSII_ASV_77"  "zuzana-RSII_ASV_78" 
 [79] "zuzana-RSII_ASV_79"  "zuzana-RSII_ASV_80"  "zuzana-RSII_ASV_81" 
 [82] "zuzana-RSII_ASV_82"  "zuzana-RSII_ASV_83"  "zuzana-RSII_ASV_84" 
 [85] "zuzana-RSII_ASV_85"  "zuzana-RSII_ASV_86"  "zuzana-RSII_ASV_87" 
 [88] "zuzana-RSII_ASV_88"  "zuzana-RSII_ASV_89"  "zuzana-RSII_ASV_90" 
 [91] "zuzana-RSII_ASV_91"  "zuzana-RSII_ASV_92"  "zuzana-RSII_ASV_93" 
 [94] "zuzana-RSII_ASV_94"  "zuzana-RSII_ASV_95"  "zuzana-RSII_ASV_96" 
 [97] "zuzana-RSII_ASV_97"  "zuzana-RSII_ASV_98"  "zuzana-RSII_ASV_99" 
[100] "zuzana-RSII_ASV_100" "zuzana-RSII_ASV_101" "zuzana-RSII_ASV_102"
[103] "zuzana-RSII_ASV_103" "zuzana-RSII_ASV_104" "zuzana-RSII_ASV_105"
[106] "zuzana-RSII_ASV_106" "zuzana-RSII_ASV_107" "zuzana-RSII_ASV_108"
[109] "zuzana-RSII_ASV_109"

[1] "A_leptoticha_CR312_1A_cut"         "A_leptoticha_CR312_1B_cut"        
 [3] "A_leptoticha_CR312_1C_cut"         "Ac_koskei_WV685_1A_cut"           
 [5] "Ac_koskei_WV685_1B_cut"            "Ac_koskei_WV685_1C_cut"           
 [7] "Ac_laevis_BEG13_1E_cut"            "Ac_laevis_BEG13_5_2_cut"          
 [9] "Ac_laevis_BEG13_5_cut"             "Ac_Laevis_BEG242_1D_cut"          
[11] "Ac_Laevis_BEG242_1F_cut"           "Ac_Laevis_BEG242_1G_cut"          
[13] "Ac_Laevis_BEG26_1_3_cut"           "Ac_Laevis_BEG26_1C_cut"           
[15] "Ac_Laevis_BEG26_1D_cut"            "Ac_morrowiae_BR983A_1A_cut"       
[17] "Ac_morrowiae_BR983A_1B_cut"        "Ac_morrowiae_BR983A_5A_cut"       
[19] "Am_fennica_fromAtt555_36_10g_cut"  "Am_gerdemannii_MT106_1C_cut"      
[21] "Am_gerdemannii_MT106_5A_cut"       "Am_gerdemannii_MT106_5B_cut"      
[23] "Am_gerdemannii_ON205A_1B_cut"      "Am_gerdemannii_ON205A_1B_SV_cut"  
[25] "Am_gerdemannii_ON205A_1C_SV_cut"   "Cl_claroideum_BEG23fromKrc_5B_cut"
[27] "Cl_etunicatum_BEG247_1A_cut"       "Cl_etunicatum_BEG247_1C_cut"      
[29] "Cl_etunicatum_BEG247_5B_cut"       "Cl_Etunicatum_IN101_10B_cut"      
[31] "Cl_Etunicatum_IN101_1A_cut"        "Cl_Etunicatum_IN101_1B_cut"       
[33] "Cl_Luteum_CR308_1A_cut"            "Cl_Luteum_CR308_1C_cut"           
[35] "Cl_Luteum_CR308_5A_cut"            "Cl_luteum_SA101_10A_cut"          
[37] "De_heterogama_BEG35_10_cut"        "De_heterogama_BEG35_5_cut"        
[39] "De_heterogama_BR154_1A_cut"        "De_heterogama_BR154_1B_cut"       
[41] "De_heterogama_BR154_1C_cut"        "De_heterogama_IL203A_1A_cut"      
[43] "De_heterogama_IL203A_1B_cut"       "De_heterogama_IL203A_1C_cut"      
[45] "Di_epigaea_KS210_1A_cut"           "Di_epigaea_KS210_1C_cut"          
[47] "Di_epigaea_KS210_5A_cut"           "Di_spurca_W6227_10_cut"           
[49] "Di_spurca_W6227_5C4_cut"           "Di_spurca_W6227_5C5_cut"          
[51] "En_infrequens_Blaszk3_10A_cut"     "Fu_mosseae_BEG12_1A_cut"          
[53] "Fu_mosseae_BEG12_1B_cut"           "Fu_mosseae_BEG12_1C_cut"          
[55] "Gi_rosea_BEG9_10_cut"              "Gi_rosea_BEG9_5C1_cut"            
[57] "Gi_rosea_BR155B_1A_cut"            "Gi_rosea_BR155B_1B_cut"           
[59] "Gi_rosea_BR155B_1C_cut"            "Gi_rosea_KS885_1A_cut"            
[61] "Gi_rosea_KS885_1B_cut"             "Gi_rosea_KS885_1C_cut"            
[63] "Ka_bistrata_Blaszk7_10B_cut"       "Pa_brasilianum_BEG239_5A_cut"     
[65] "Pa_brasilianum_BR105_1C_cut"       "Pa_Brasilianum_FL710_1C_cut"      
[67] "Pa_Brasilianum_FL710_5A_cut"       "Pa_occultum_CU126_1B_cut"         
[69] "Pa_occultum_CU126_1C_cut"          "Pa_occultum_CU126_5A_cut"         
[71] "Pa_scintillans_Blaszk6_5B_cut"     "Ra_gregaria_BEG243_1A_cut"        
[73] "Ra_gregaria_BEG243_1B_cut"         "Ra_gregaria_BEG243_1C_cut"        
[75] "Rh_Irregularis_Chomutov_10B_cut"   "Rh_Irregularis_Chomutov_1A_cut"   
[77] "Rh_irregularis_PH5_1C_cut"         "Sac_cerevisiae_cut"               
[79] "Sc_calospora_AU212B_10B_cut"       "Sc_calospora_AU212B_5A_cut"       
[81] "Sc_calospora_BEG245_10A_cut"       "Sc_calospora_BEG245_1B_cut"       
[83] "Sc_calospora_BEG245_5A_cut"        "SccalosporaAU212B1C_cut"          
[85] "Se_deserticola_NC302A_1A_cut"      "Se_deserticola_NC302A_1C_cut"     
[87] "Se_deserticola_NC302A_5A_cut"

In [14]:
bios$writeXStringSet(rep_seqs, "zuzana/zuzana_RSII_rep_seqs.fasta")
write.table(
  asv_table,
  "zuzana/zuzana_RSII_asv_table.tsv",
  sep = "\t",
  row.names = TRUE,
  col.names = NA,
  quote = FALSE
)